# 38장 · 재정렬과 다중 질의 검색

재정렬은 기존 후보들의 순서를 바꾸고, 다중 질의 검색은 질문 표현을 늘려 후보를 모읍니다. 첫 검색에 정답 문서가 없는 경우 두 방법의 차이가 드러납니다.

대출 기간을 묻는 질문에서 마침표만 담은 청크가 1위가 됐습니다. 기존 후보를 다시 정렬하는 방법과 질의를 확장해 새 후보를 가져오는 방법을 같은 자료에서 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## sorted()를 이용한 후보 재정렬

37장의 16글자 청크 중 library-1:0에는 14일이 있고 library-1:1에는 마침표만 있습니다. 순위 차이를 눈으로 보기 위해 이 두 입력의 순서를 뒤집어 둡니다. 여기서 쓰는 점수는 수업용 단어 일치이며 학습된 재순위 모델의 결과와 구분합니다.

**후보 집합과 재정렬 순위**　재정렬 전 첫 후보는 마침표 하나입니다. 대출 문장을 앞으로 보내도 전체 후보 수는 같습니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter38/index.html#practice-space)


### 셀 01 · 재정렬 전 후보 순위

후보 두 건을 그대로 받아 첫 번째 청크의 본문을 읽습니다.


In [1]:
candidates = [{'id': 'library-1:1', 'text': '.', 'source': '가상 도서관 이용 안내', 'page': 1},
 {'id': 'library-1:0',
  'text': '도서 대출 기간은 14일입니다',
  'source': '가상 도서관 이용 안내',
  'page': 1}]
print(candidates[0]['id'], repr(candidates[0]['text']))

library-1:1 '.'


마침표만 있는 library-1:1이 보입니다. 원문은 맞지만 이 조각만 받은 답변 함수에는 기간을 확인할 숫자가 없습니다.


### 셀 02 · sorted()의 key와 reverse

각 후보의 본문에 ‘대출’이 들어 있는지 검사해 해당 문장을 앞으로 보냅니다.


In [2]:
ranked = sorted(candidates, key=lambda doc: '대출' in doc['text'], reverse=True)
print(ranked[0]['id'], ranked[0]['text'])

library-1:0 도서 대출 기간은 14일입니다


‘대출’을 포함하는 library-1:0이 2위에서 1위로 올라왔습니다.

재정렬의 효과를 확인하려면 후보 목록에서 library-1:0을 잠시 빼 보세요. 남은 마침표를 아무리 정렬해도 ‘14일’은 돌아오지 않습니다. 이 셀의 점수는 ‘대출’이 들어 있는지를 나타내는 불리언 값입니다. 학습된 재정렬 모델이 질문과 문장의 뜻을 얼마나 잘 비교하는지는 별도 실험에서 측정해야 합니다.

<div class="example-table"><table><caption>같은 후보 두 건의 재정렬</caption><thead><tr><th scope="col">청크 ID</th><th scope="col">청크 본문</th><th scope="col">검색 순위</th><th scope="col">키워드 정렬 후</th></tr></thead><tbody><tr><td>library-1:1</td><td>.</td><td>1</td><td>2</td></tr><tr><td>library-1:0</td><td>도서 대출 기간은 14일입니다</td><td>2</td><td>1</td></tr></tbody></table></div>


#### 재정렬로 고칠 수 있는 범위

<mark>재정렬은 이미 뽑힌 후보의 순서만 바꿉니다.</mark> 마침표 한 글자짜리 조각과 대출 기간이 들어간 조각 중 무엇이 앞으로 왔는지 보세요. 정답 청크가 후보에 없다면 순위 기준을 바꿔도 그 청크는 등장하지 않습니다.

선수 개념 · 37장 청크 크기·중첩·top-k 비교 실험

#### 마침표와 대출 문장의 순서가 뒤집히는 이유

```mermaid
flowchart TD
ninitial["입력 순위"]
  nkey["정렬 키"]
  nsort["내림차순"]
  ninitial --> nkey
  nkey --> nsort
```

첫 검색 결과를 그대로 고정하고 각 본문에 ‘대출’이 들어 있는지만 정렬 키로 사용합니다.

## 질의 확장과 청크 ID 중복 제거

‘책을 빌릴 수 있는 기간은?’을 세 낱말 벡터로 바꾸면 대출 항목이 0입니다. 이번에는 검색용 표현 ‘대출 기간’을 직접 추가합니다. 원래 질문도 남겨 두고, 공백만 다른 표현은 한 번만 검색합니다.


### 과제 03 · 중복 질의 제거

원래 질문과 확장문을 순서대로 읽어 공백을 정리하고 중복 없이 남기세요.


In [3]:
queries = ['책을 빌릴 수 있는 기간은?', '대출 기간', '대출   기간']
# 첫 등장 순서를 지키며 중복을 제거하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>queries = [&#x27;책을 빌릴 수 있는 기간은?&#x27;, &#x27;대출 기간&#x27;, &#x27;대출   기간&#x27;]
unique = []
for query in queries:
    clean = &#x27; &#x27;.join(query.split())
    if clean not in unique:
        unique.append(clean)
print(unique)</code></pre>
<p>실행 결과</p><pre><code>[&#x27;책을 빌릴 수 있는 기간은?&#x27;, &#x27;대출 기간&#x27;]</code></pre>
<p>두 검색문이 남습니다. 검색문 중복을 줄였어도 양쪽 검색 결과에 같은 청크가 나올 수 있습니다. 결과를 합칠 때는 청크 ID 기준으로 한 번 더 확인합니다.</p>
</details>


### 검색어가 달라지면 후보를 다시 가져옵니다

<p>과제의 첫 질의 ‘책을 빌릴 수 있는 기간은?’은 검색 함수가 아는 단어를 포함하지 않습니다. ‘대출 기간’을 함께 보내면 첫 질의에서 빠진 문서를 찾을 수 있습니다. 셀 04에서는 두 질의 모두 ‘대출’을 포함하도록 바꿔, 같은 청크가 돌아오는 경우를 따로 실험합니다.</p>


### 무엇을 기준으로 중복을 판단할까요?

<p>요청을 보내기 전에는 공백을 정리한 검색문을 비교합니다. 결과를 합칠 때는 청크 ID를 비교합니다. 본문만 비교하면 서로 다른 쪽에 있는 같은 문장을 합쳐 출처 하나를 잃을 수 있습니다. 셀 04는 ID만 모으므로 같은 ID의 원문 충돌까지 검사하지는 않습니다.</p>


### 셀 04 · 검색 결과의 청크 ID 합집합

같은 고정 인덱스에서 검색문별 결과를 받아 첫 등장 순서대로 합칩니다. 이 예의 검색은 ‘대출’ 포함 여부로 찾는 작은 로컬 검색입니다.

과제 03에서는 ‘책을 빌릴 수 있는 기간은?’을 출발점으로 표현을 늘렸습니다. 이번 셀은 두 표현 모두 ‘대출’을 포함하는 별도 예입니다. 여기서 관찰할 것은 새 문서 발견이 아니라 두 검색이 돌려준 같은 ID를 한 번만 남기는 과정입니다.


In [4]:
index = [{'id': 'library-1:0',
  'text': '도서 대출 기간은 14일입니다',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-1:1', 'text': '.', 'source': '가상 도서관 이용 안내', 'page': 1}]
search_queries = ['대출 기간은 며칠인가요?', '대출 기간']

def find_chunks(query):
    return [row['id'] for row in index if '대출' in query and '대출' in row['text']]

batches = [find_chunks(query) for query in search_queries]
merged = list(dict.fromkeys(chunk for batch in batches for chunk in batch))
print(batches)
print(merged)

[['library-1:0'], ['library-1:0']]
['library-1:0']


질문 표현 두 개가 같은 library-1:0을 찾았습니다. 검색은 두 번 했지만 답의 근거에는 청크 하나만 넘기면 됩니다. 원래 후보에 없던 청크가 추가되려면 확장한 검색문에서 실제로 다른 결과를 얻어야 합니다.


### 과제 05 · 정답 청크가 후보에 없을 때

마침표만 남은 후보를 같은 규칙으로 재정렬하세요. 이어 원래 표현과 ‘대출 기간’을 각각 find_chunks에 넣으세요. ‘14일’이 들어 있는 청크를 되찾은 단계가 무엇인지 기록하세요.


In [ ]:
pool = candidates[:1]
# 주어진 후보만 정렬한 결과, 원래 질의의 검색 결과, 확장 질의의 검색 결과를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>pool = candidates[:1]
reranked = sorted(pool, key=lambda doc: &#x27;대출&#x27; in doc[&#x27;text&#x27;], reverse=True)
print([doc[&#x27;id&#x27;] for doc in reranked])
print(find_chunks(&#x27;책을 빌릴 수 있는 기간은?&#x27;))
print(find_chunks(&#x27;대출 기간&#x27;))</code></pre>
<p>실행 결과</p><pre><code>[&#x27;library-1:1&#x27;]
[]
[&#x27;library-1:0&#x27;]</code></pre>
<p>첫 줄은 마침표였던 후보를 그대로 반환합니다. 둘째 줄에는 ‘대출’이 없어 결과가 비었습니다. 셋째 줄은 인덱스를 다시 검색했기 때문에 후보에 없던 library-1:0을 가져왔습니다. 원래 표현과 확장 표현을 모두 실행하면 검색 함수 호출은 두 번입니다.</p>
</details>


## 돌아보기

키워드 재순위가 성공한 위 사례는 의도적으로 순서를 뒤집은 연습입니다. 실제 채택 여부는 다음 장의 같은 평가 질문에서 나온 결과와 추가 작업 횟수로 판단하세요.


## 실제 재정렬기가 ‘예약 3일’을 ‘대출 14일’보다 높게 고른 경우

도서관을 이용하는 사람은 “대출 기간” 대신 “책을 빌리면 며칠 동안 볼 수 있나요?”라고 묻기도 합니다. 점 하나뿐인 조각과 대출 안내에 예약 도서 공지를 한 건 더 넣고, 같은 세 후보를 실제 CrossEncoder가 어떤 순서로 골랐는지 읽습니다.

### 셀 07 · 실측한 점수를 원래 후보 ID와 짝짓기

BAAI/bge-reranker-v2-m3에 (질문, 후보 본문) 세 쌍을 넣어 CPU에서 얻은 점수입니다. 기본 셀은 저장한 값을 읽으므로 모델 없이도 실행할 수 있습니다. 후보를 정렬하기 전에는 점수가 들어온 입력 순서를 유지하세요.

In [5]:
pool_for_rerank = [*candidates, {"id": "reserve-1", "text": "예약한 책은 도착 안내를 받은 뒤 3일 이내에 찾아가세요."}]
reader_question = "책을 빌리면 며칠 동안 볼 수 있나요?"
catalog_question = "도서 대출 기간은 며칠입니까?"
measured_scores = {
    reader_question: [0.0000324154, 0.0150453150, 0.0165906642],
    catalog_question: [0.0000296412, 0.9921278954, 0.0099714696],
}
for row, score in zip(pool_for_rerank, measured_scores[reader_question], strict=True):
    print(row["id"], f"{score:.5f}")

library-1:1 0.00003
library-1:0 0.01505
reserve-1 0.01659


일상 표현 질문에서 예약 공지 reserve-1의 점수가 대출 문서 library-1:0보다 조금 높습니다. 둘 다 책과 기간을 말하지만 3일은 예약 도서를 찾아갈 기한입니다. 이 예에서 가장 높은 점수가 곧 정답이라는 결론은 맞지 않습니다.

### 과제 08 · 질문 표현만 바꾸고 기대 근거의 순위 검산하기

질문마다 해당 질문의 실측 점수로 후보를 내림차순 정렬하세요. 각 질문의 1위 ID와 정답 library-1:0의 순위(1부터 시작)를 출력합니다. 같은 후보를 유지해야 질의 표현의 영향만 볼 수 있습니다.

In [6]:
# 함수 ranking_for(question)는 pool_for_rerank와 그 질문의 measured_scores를 사용합니다.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def ranking_for(question):
    scored = zip(pool_for_rerank, measured_scores[question], strict=True)
    ordered = sorted(scored, key=lambda pair: pair[1], reverse=True)
    return [row[&quot;id&quot;] for row, score in ordered]

for query in (reader_question, catalog_question):
    ordered = ranking_for(query)
    print(ordered[0], ordered.index(&quot;library-1:0&quot;) + 1)</code></pre>
<p>실행 결과</p><pre><code>reserve-1 2
library-1:0 1</code></pre>
<p>일상 표현에서는 reserve-1이 1위이고 기대 근거는 2위입니다. 대출 기간을 명시한 표현에서는 기대 근거가 1위입니다. 질의 확장으로 후자의 검색 결과를 더할 수 있지만, 두 질의로 후보를 찾는 횟수와 최종 정렬 점수는 별도로 기록해야 합니다.</p>
</details>

### 셀 09 · 선택 실습 · 로컬 CrossEncoder로 세 쌍 다시 채점하기

sentence-transformers와 해당 모델 파일을 준비한 수업 커널에서 model_dir에 로컬 모델 폴더를 지정하세요. 정확한 점수는 환경에 따라 조금 달라질 수 있으므로 먼저 상위 ID를 비교합니다. 모델을 준비하기 전에는 이 셀이 외부에서 내려받지 않습니다.

In [7]:
from pathlib import Path
model_dir = None  # 모델을 준비했다면 Path("수업에서 준비한 모델 폴더")로 변경
if model_dir is None:
    print("위 예제는 저장된 실측 점수입니다. 모델을 연결하면 다시 계산합니다.")
else:
    from sentence_transformers import CrossEncoder
    encoder = CrossEncoder(str(Path(model_dir)), local_files_only=True, device="cpu")
    for query in (reader_question, catalog_question):
        pairs = [(query, row["text"]) for row in pool_for_rerank]
        scores = encoder.predict(pairs, show_progress_bar=False)
        best = max(range(len(scores)), key=lambda pos: float(scores[pos]))
        print(pool_for_rerank[best]["id"], round(float(scores[best]), 5))

위 예제는 저장된 실측 점수입니다. 모델을 연결하면 다시 계산합니다.


이 셀에서 실행하는 predict는 실제 후보 문장과 질의를 한 쌍씩 함께 읽습니다. 실측에는 sentence-transformers 6.1.0과 모델 리비전 953dc6f6의 로컬 파일을 썼습니다. 기본 안내만 출력되었다면 이번 커널에서는 재채점하지 않은 상태입니다.

<mark>재정렬기는 검색된 후보 안에서만 순서를 바꾸며, 남은 후보 중에서도 틀릴 수 있습니다.</mark> 반납 질문의 근거가 애초에 pool_for_rerank에 없다면 예제의 세 점수를 어떻게 정렬해도 그 안내문은 나오지 않습니다. 근거가 후보에 있는데 2위가 된 대출 질문과 따로 기록하세요.

39장에서는 기대 근거가 실제로 올라간 문항과 새로 틀린 문항을 기준선과 나란히 놓고 채택 여부를 정합니다.